<a href="https://colab.research.google.com/github/Husan2/NTU_GAI/blob/main/0225%E4%BD%9C%E6%A5%AD%E7%A5%9E%E7%B6%93%E7%B6%B2%E8%B7%AF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 類神經網路 GOGOGO
## 目標：建立 5 層深度學習模型，來進行 MNIST 手寫數字辨識。
- **網路架構**：5 層神經網路，加入 `Dropout` 以防止過擬合。
- **激活函數選擇**：測試不同函數（ReLU、Sigmoid、Tanh），最終選擇 ReLU 搭配 Softmax。
- **超參數測試**：嘗試不同學習率、批次大小、最佳化器，並記錄影響。

# 1. 下載必要套件以及數據集

In [2]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
to

In [8]:
%matplotlib inline

# 標準數據分析、畫圖套件
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 神經網路方面
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam

# 互動設計用
from ipywidgets import interact_manual

# 神速打造 web app 的 Gradio
import gradio as gr

## 2. 讀入 MNIST 數據庫

In [4]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f'訓練資料總筆數: {len(x_train)}')
print(f'測試資料總筆數: {len(x_test)}')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
訓練資料總筆數: 60000
測試資料總筆數: 10000


# 3. 資料前處理
- 影像展平成 784 維向量
- 像素值歸一化 (0~1) 以提升學習效率

In [5]:
# 正規化至 [0,1]
x_train = x_train.reshape(-1, 28*28) / 255.0
x_test = x_test.reshape(-1, 28*28) / 255.0
# 標籤轉換為 10 類別
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# 4. 建立 DNN 模型（5 層，加入 Dropout 避免過擬合）
- 使用 ReLU 作為隱藏層激活函數 (減少梯度消失問題)
- 第三層使用 tanh
- 最後一層使用 Softmax 進行多分類

- ReLU (max(0, x)) 是目前最常見的隱藏層激活函數，能有效解決梯度消失問題。
 - 比 Sigmoid 和 Tanh 具有更快的收斂速度，適合較深的神經網路。
 - 避免 Sigmoid 的梯度消失 (當輸入值過大或過小時，梯度趨近於 0)。

 *其他可選激活函數比較*
 1. **Sigmoid (σ(x) = 1 / (1 + e^(-x)))**
    - 優點：適用於輸出層 (二元分類)
    - 缺點：容易出現梯度消失，使深層網路難以學習

 2. **Tanh (tanh(x) = 2σ(2x) - 1)**
    - 優點：輸出範圍 -1 到 1，比 Sigmoid 更好 (因為均值為 0，梯度較大)
    - 缺點：仍可能在深層網路中出現梯度消失

 3. **Softmax (e^x_i / Σe^x)**
    - 適用於最後一層，確保輸出為機率分佈 (多分類問題)

In [7]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(784,)),  # 第一層
    Dense(128, activation='relu'),  # 第二層
    Dense(128, activation='tanh'),  # 第三層
    Dense(64, activation='relu'),  # 第四層
    Dense(10, activation='softmax')  # 第五層（輸出層）
])

- 使用 Adam() 取代 SGD()，加快收斂速度，避免梯度震盪問題。
- learning_rate 設為 0.001

In [9]:
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                      │ (None, 64)                  │          50,240 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 83,978 (328.04 KB)

 Trainable params: 83,978 (328.04 KB)

 Non-trainable params: 0 (0.00 B)

# 6. 訓練模型

In [11]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=20, batch_size=100)

Epoch 1/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8287 - loss: 0.5853 - val_accuracy: 0.9557 - val_loss: 0.1437
Epoch 2/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9631 - loss: 0.1246 - val_accuracy: 0.9689 - val_loss: 0.1027
Epoch 3/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9737 - loss: 0.0848 - val_accuracy: 0.9663 - val_loss: 0.1041
Epoch 4/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9794 - loss: 0.0656 - val_accuracy: 0.9752 - val_loss: 0.0877
Epoch 5/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9814 - loss: 0.0576 - val_accuracy: 0.9705 - val_loss: 0.0992
Epoch 6/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9856 - loss: 0.0453 - val_accuracy: 0.9726 - val_loss: 0.0976
Epoch 7/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9872 - loss: 0.0400 - val_accuracy: 0.9743 - val_loss: 0.0913
Epoch 8/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9903 - loss: 0.0303 - val_accuracy: 0.

# 看結果

In [12]:
loss, accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9727 - loss: 0.1329


In [13]:
print(f'測試資料的準確度為 {accuracy*100:.2f}%')

測試資料的準確度為 97.73%


## 激活函數測試與比較

| 測試 | 激活函數 | 最佳化器 | 驗證準確率 |
|------|------------|-------|----------|
| 測試 1 | ReLU       | SGD   | 85%  |
| 測試 2 | ReLU       | Adam  | 97%  |
| 測試 3 | Sigmoid    | Adam  | 92%  |
| 測試 4 | Tanh       | Adam  | 95%  |

- 最後決定第1.2.4使用ReLU，第三層使用Tanh，準確率為97.73%

# 7.用Gradio來展示
- 關於程式碼細部在筆記中有紀錄了，這裡就先不展示


In [15]:
def resize_image(inp):
    image = np.array(inp["layers"][0], dtype=np.float32)
    image = image.astype(np.uint8)
    image_pil = Image.fromarray(image)
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3])
    image_pil = background
    image_gray = image_pil.convert("L")
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))
    img_array = 255 - img_array
    img_array = img_array.reshape(1, 784) / 255.0

    return img_array

In [17]:
def recognize_digit(inp):
    img_array = resize_image(inp)
    prediction = model.predict(img_array).flatten()
    labels = list('0123456789')
    return {labels[i]:float(prediction[i]) for i in range(10)}

In [18]:
ifact = gr.Interface(
    fn = recognize_digit,
    inputs = gr.Sketchpad(),
    outputs = gr.Label(num_top_classes=3),
    title = 'MNIST 手寫數字辨識',
    description = '請在畫板上繪畫數字',
)

ifact.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d3c33e06c699f30f6f.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d3c33e06c699f30f6f.gradio.live
